# Public ASX Publication-Derived Research

This notebook runs a public approximation of the publication research workflow on the Community/Team Yahoo ASX lake.

Important limitations:
- The security universe is the configured public Yahoo/yFinance universe, not point-in-time historical ASX 200 membership.
- STW.AX is used as a public benchmark proxy.
- Yahoo/yFinance data quality and delisting coverage are weaker than the licensed Norgate publication dataset.
- Sector classifications are the current Yahoo-derived curated reference map, not historical point-in-time sector classifications.
- Results from this notebook must not be described as a replication of the publication empirical dataset.


In [ ]:
from __future__ import annotations

import gc
import io
import os

import boto3
import pandas as pd
import pyarrow.parquet as pq

from asx_research_panel import build_asx_research_panel, summarize_asx_research_panel_quality
from rba_cash_rate_tri_local import RbaCashRateTriConfig, load_rba_cash_rate_tri
from stw_benchmark_local import STWIngestionConfig, download_stw_history, resolve_date_window

MINIO_ENDPOINT = os.environ.get("S3_ENDPOINT_URL", "http://minio:9000")
MINIO_ACCESS_KEY = os.environ["AWS_ACCESS_KEY_ID"]
MINIO_SECRET_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]
MINIO_REGION = os.environ.get("AWS_DEFAULT_REGION", "local-01")

CURATED_BUCKET = "curated"
CURATED_KEY = "tabular/market_ohlcv_daily_v2/exchange=ASX/asx_ohlcv_panel_curated.parquet"
SECTOR_MAP_KEY = "tabular/asx_ticker_sector_map_v1/exchange=ASX/asx_ticker_sector_map.parquet"
CURATED_COLUMNS = ["ticker", "trade_date", "close", "adj_close", "volume"]

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name=MINIO_REGION,
)


In [ ]:
response = s3.get_object(Bucket=CURATED_BUCKET, Key=CURATED_KEY)
table = pq.read_table(io.BytesIO(response["Body"].read()), columns=CURATED_COLUMNS)
curated = table.to_pandas()
del table, response

curated["ticker"] = curated["ticker"].astype("category")
curated["close"] = pd.to_numeric(curated["close"], errors="coerce").astype("float32")
curated["adj_close"] = pd.to_numeric(curated["adj_close"], errors="coerce").astype("float32")
curated["volume"] = pd.to_numeric(curated["volume"], errors="coerce").astype("float32")
curated["trade_date"] = pd.to_datetime(curated["trade_date"], errors="coerce")
curated = curated.dropna(subset=["trade_date"]).sort_values(["ticker", "trade_date"]).reset_index(drop=True)
gc.collect()

{
    "rows": len(curated),
    "tickers": curated["ticker"].nunique(),
    "date_min": curated["trade_date"].min(),
    "date_max": curated["trade_date"].max(),
}


In [ ]:
sector_response = s3.get_object(Bucket=CURATED_BUCKET, Key=SECTOR_MAP_KEY)
sector_table = pq.read_table(io.BytesIO(sector_response["Body"].read()))
sector_map = sector_table.to_pandas()

{
    "rows": len(sector_map),
    "resolved": int((sector_map["status"] == "resolved").sum()),
    "review": int((sector_map["status"] != "resolved").sum()),
    "sectors": int(sector_map["sector"].replace("", pd.NA).nunique(dropna=True)),
}


In [ ]:
research_panel = build_asx_research_panel(curated, min_history=1)
del curated
gc.collect()

{
    "rows": len(research_panel),
    "tickers": research_panel["ticker"].nunique(),
    "date_min": research_panel["trade_date"].min(),
    "date_max": research_panel["trade_date"].max(),
    "universe_treatment": research_panel.attrs.get("universe_treatment"),
    "liquidity_proxy": research_panel.attrs.get("liquidity_proxy"),
}


In [ ]:
research_panel_quality = summarize_asx_research_panel_quality(research_panel)

research_panel_quality


In [ ]:
start_date, end_date = resolve_date_window(research_panel)

stw = download_stw_history(
    STWIngestionConfig(
        vendor_symbol="STW.AX",
        start_date=start_date,
        end_date=end_date,
    )
)

{
    "rows": len(stw),
    "date_min": stw["trade_date"].min(),
    "date_max": stw["trade_date"].max(),
    "missing_benchmark_returns": int(stw["benchmark_return"].isna().sum()),
}


In [ ]:
rba = load_rba_cash_rate_tri(
    RbaCashRateTriConfig(
        start_date=start_date,
        end_date=end_date,
    )
)

{
    "rows": len(rba),
    "date_min": rba["trade_date"].min(),
    "date_max": rba["trade_date"].max(),
    "missing_risk_free_returns": int(rba["risk_free_return"].isna().sum()),
}


## Trend following

Publication-derived walk-forward:
- 3-year formation window
- 1-year evaluation window
- 1-year step
- benchmark-relative parameter selection
- ex-ante liquidity-tier transaction costs


In [ ]:
from strategies.public_walk_forward import run_public_walk_forward

trend_result = run_public_walk_forward(
    strategy_name="trend_following",
    prices=research_panel,
    benchmark_returns=stw[["trade_date", "benchmark_return"]],
    risk_free_returns=rba[["trade_date", "risk_free_return"]],
)

trend_result.fold_table


In [ ]:
trend_result.fold_summary


## Mean reversion


In [ ]:
mean_reversion_result = run_public_walk_forward(
    strategy_name="mean_reversion",
    prices=research_panel,
    benchmark_returns=stw[["trade_date", "benchmark_return"]],
    risk_free_returns=rba[["trade_date", "risk_free_return"]],
)

mean_reversion_result.fold_table


In [ ]:
mean_reversion_result.fold_summary


## Pairs trading

Publication-derived pair formation uses the top 20 liquid candidates, prefers same-sector pairs using the curated Yahoo-derived sector reference map, applies the residual-stationarity screening threshold of 0.05, selects up to five pairs, normalizes gross capital across both legs, and applies liquidity-tier weighted transaction costs. Cross-sector candidates are used only when the same-sector candidate pool is insufficient.


In [ ]:
from strategies.public_pairs import run_public_pairs_walk_forward

pairs_result = run_public_pairs_walk_forward(
    prices=research_panel,
    benchmark_returns=stw[["trade_date", "benchmark_return"]],
    sector_map=sector_map,
    risk_free_returns=rba[["trade_date", "risk_free_return"]],
)

pairs_result.fold_table


In [ ]:
pairs_result.fold_summary


In [ ]:
pairs_result.pair_diagnostics[[
    "fold_id", "pair_id", "left_ticker", "right_ticker",
    "left_sector", "right_sector", "cointegration_pvalue",
    "hedge_ratio", "formation_observations", "candidate_source",
    "chosen_parameters"
]]


### Walk-forward fold effects

The chart below shows evaluation-fold `net_excess_nav_difference` for the three walk-forward strategies. It is descriptive fold-level evidence and does not replace the inference tests below.


In [ ]:
import matplotlib.pyplot as plt

fold_effects = pd.concat(
    [
        trend_result.fold_summary[["fold_id", "net_excess_nav_difference"]].assign(strategy="Trend Following"),
        mean_reversion_result.fold_summary[["fold_id", "net_excess_nav_difference"]].assign(strategy="Mean Reversion"),
        pairs_result.fold_summary[["fold_id", "net_excess_nav_difference"]].assign(strategy="Pairs Trading"),
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(11, 5))
for strategy, group in fold_effects.groupby("strategy", sort=False):
    ax.plot(group["fold_id"], group["net_excess_nav_difference"], marker="o", label=strategy)

ax.axhline(0.0, linewidth=1)
ax.set_title("Walk-forward evaluation-fold effects")
ax.set_xlabel("Evaluation fold")
ax.set_ylabel("Net excess NAV difference")
ax.legend()
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()


## Tax-loss selling

Publication-derived Australian financial-year-end event study:
- selects the bottom decile of trailing 252-trading-day performers
- selection occurs 11 trading days before the event date
- event date is the last trading day on or before 30 June
- compares a ±10-trading-day event window with a matched window centred 60 trading days earlier
- requires complete security and STW benchmark windows
- applies ex-ante liquidity-tier round-trip transaction costs symmetrically to event and control observations

The public Yahoo universe is retrospective and does not reproduce point-in-time ASX 200 membership. Statistical inference and year-robustness are intentionally deferred to the separate inference stage.


In [ ]:
from strategies.public_tax_loss import run_public_tax_loss_event_study

tax_loss_result = run_public_tax_loss_event_study(
    prices=research_panel,
    benchmark_returns=stw[["trade_date", "benchmark_return"]],
)

tax_loss_result.summary


In [ ]:
tax_loss_result.event_study[[
    "year", "ticker", "selection_date", "event_date", "control_date",
    "selection_trailing_12m_return", "liquidity_tier",
    "net_return_difference", "abnormal_net_return_difference",
    "complete_event_window", "complete_control_window",
]].head(20)


In [ ]:
tax_loss_result.liquidity_diagnostics.head(20)


## Statistical inference

Publication-derived confirmatory inference:
- Trend Following, Mean Reversion, and Pairs Trading use evaluation-fold `net_excess_nav_difference`.
- Tax-Loss Selling uses equal-weight calendar-year means of complete benchmark-adjusted `abnormal_net_return_difference` observations.
- Mean effects use a deterministic bootstrap 95% confidence interval.
- One-sided sign-flip tests assess whether the mean effect is greater than zero.
- Holm adjustment is applied across the four primary strategy hypotheses.

These tests assess transferability of the publication-derived methodology on the public Yahoo/STW proxy dataset. They do not convert the public dataset into a replication of the licensed publication sample.


In [ ]:
from strategies.inference import build_public_primary_inference

inference_result = build_public_primary_inference(
    trend_summary=trend_result.fold_summary,
    mean_reversion_summary=mean_reversion_result.fold_summary,
    pairs_summary=pairs_result.fold_summary,
    tax_loss_event_study=tax_loss_result.event_study,
)

inference_result.primary_inference[[
    "analysis_key",
    "effect_estimate",
    "ci_lower_95",
    "ci_upper_95",
    "p_value",
    "adjusted_p_value",
    "reject_null_0_05",
    "sample_size",
    "sample_unit",
    "primary_metric",
    "claim_label",
]]


### Primary strategy effects with 95% confidence intervals

The confidence intervals and effect estimates below are the same values reported in the primary inference table.


In [ ]:
primary_plot = inference_result.primary_inference.copy()
primary_plot["strategy"] = primary_plot["analysis_key"].map(
    {
        "trend_following": "Trend Following",
        "mean_reversion": "Mean Reversion",
        "pairs_trading": "Pairs Trading",
        "tax_loss_selling": "Tax-Loss Selling",
    }
)

x = list(range(len(primary_plot)))
lower_error = primary_plot["effect_estimate"] - primary_plot["ci_lower_95"]
upper_error = primary_plot["ci_upper_95"] - primary_plot["effect_estimate"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    x,
    primary_plot["effect_estimate"],
    yerr=[lower_error, upper_error],
    fmt="o",
    capsize=5,
)
ax.axhline(0.0, linewidth=1)
ax.set_xticks(x, primary_plot["strategy"])
ax.set_title("Public ASX primary strategy effects")
ax.set_ylabel("Primary effect estimate")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()


In [ ]:
inference_result.tax_loss_year_effects


### Tax-Loss annual abnormal effects

The annual series is the same calendar-year aggregation used as the Tax-Loss inference sample.


In [ ]:
tax_year_plot = inference_result.tax_loss_year_effects.sort_values("year")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(tax_year_plot["year"].astype(str), tax_year_plot["mean_abnormal_net_return_difference"])
ax.axhline(0.0, linewidth=1)
ax.set_title("Tax-Loss annual abnormal net return difference")
ax.set_xlabel("Calendar year")
ax.set_ylabel("Mean abnormal net return difference")
fig.tight_layout()
plt.show()


## Public data vs publication evidence

This section compares the newly computed Yahoo/STW inference with frozen aggregate evidence from the publication-facing reproducibility repository. The comparison is descriptive evidence of data/design sensitivity, not a pooled test and not an attempt to reproduce the licensed Norgate sample.

The publication article's principal mechanism evidence is a controlled same-vendor Norgate experiment showing that retrospective historical-universe construction materially changes trend-following performance. That mechanism evidence is shown separately from the four-strategy confirmatory comparison.


In [ ]:
from strategies.publication_comparison import build_publication_comparison

publication_comparison = build_publication_comparison(inference_result.primary_inference)

publication_comparison[[
    "analysis_key",
    "public_effect_estimate",
    "publication_effect_estimate",
    "public_adjusted_p_value",
    "publication_adjusted_p_value",
    "public_supported_after_holm",
    "publication_supported_after_holm",
    "support_changed",
    "public_effect_direction",
    "publication_effect_direction",
    "effect_direction_changed",
    "public_sample_size",
    "publication_sample_size",
    "public_sample_unit",
    "publication_sample_unit",
]]


### Public Yahoo/STW versus publication/Norgate effects

The two series are shown descriptively to make the change in empirical conclusions visible. They remain separate datasets and are not pooled.


In [ ]:
comparison_plot = publication_comparison.copy()
comparison_plot["strategy"] = comparison_plot["analysis_key"].map(
    {
        "trend_following": "Trend Following",
        "mean_reversion": "Mean Reversion",
        "pairs_trading": "Pairs Trading",
        "tax_loss_selling": "Tax-Loss Selling",
    }
)

x = list(range(len(comparison_plot)))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    [value - width / 2 for value in x],
    comparison_plot["public_effect_estimate"],
    width=width,
    label="Public Yahoo/STW",
)
ax.bar(
    [value + width / 2 for value in x],
    comparison_plot["publication_effect_estimate"],
    width=width,
    label="Publication/Norgate",
)
ax.axhline(0.0, linewidth=1)
ax.set_xticks(x, comparison_plot["strategy"])
ax.set_title("Public versus publication primary effect estimates")
ax.set_ylabel("Primary effect estimate")
ax.legend()
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()


In [ ]:
from strategies.publication_comparison import publication_trend_mechanism_summary

publication_trend_mechanism_summary()


In [ ]:
from strategies.publication_comparison import public_vs_publication_data_design

public_vs_publication_data_design()


### Interpretation

The Yahoo/STW and Norgate publication analyses are deliberately separate empirical result sets. Different support decisions or effect directions are therefore interpreted as evidence that the empirical conclusions are sensitive to the data/design environment.

The controlled Norgate mechanism experiment provides direct evidence that retrospective universe construction can materially inflate trend-following performance within common vendor/security coverage. It does not prove that this mechanism fully explains every Yahoo-versus-Norgate difference.

Accordingly, the public Community implementation should be described as a reproducible demonstration of publication-derived methodology and data-quality sensitivity, not as an attempt to prove the exact publication results using Yahoo data.


## Strategy comparison


In [ ]:
def summarize_folds(name: str, result) -> dict[str, object]:
    summary = result.fold_summary
    return {
        "strategy": name,
        "folds": int(summary["fold_id"].nunique()),
        "average_annualized_return": float(summary["annualized_return"].mean()),
        "average_annualized_volatility": float(summary["annualized_volatility"].mean()),
        "average_sharpe_ratio": float(summary["sharpe_ratio"].mean()),
        "average_max_drawdown": float(summary["max_drawdown"].mean()),
        "average_net_excess_nav_difference": float(summary["net_excess_nav_difference"].mean()),
    }

comparison = pd.DataFrame(
    [
        summarize_folds("trend_following", trend_result),
        summarize_folds("mean_reversion", mean_reversion_result),
        summarize_folds("pairs_trading", pairs_result),
    ]
)

comparison


## Interpretation boundary

These results demonstrate publication-derived methodology on a public Yahoo/yFinance ASX dataset.

They do not reproduce:
- licensed Norgate data quality,
- permanent Norgate asset identity,
- historical point-in-time ASX 200 membership,
- historical point-in-time sector classifications,
- the publication's official benchmark series,
- or the exact publication empirical results.

The correct description is: **publication-derived methodology demonstrated on a publicly reproducible ASX dataset, with explicit comparison to publication evidence showing data/design sensitivity**.
